In [8]:
import pygame
import pymunk
import pymunk.pygame_util
import math
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO

# ==========================================
# 1. CONFIGURATION
# ==========================================
WIDTH, HEIGHT = 1000, 800
FPS = 60

# ==========================================
# 2. OPENAI GYMNASIUM WRAPPER
# ==========================================
class DriftEnv(gym.Env):
    def __init__(self):
        super(DriftEnv, self).__init__()

        # --- AI Interface ---
        # Action Space: [Steering (-1 to 1), Throttle (-1 to 1), Handbrake (0 to 1)]
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(3,), dtype=np.float32)

        # Observation Space: [5 LiDAR distances, Forward Speed, Lateral Speed]
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(7,), dtype=np.float32)

        # --- Physics Engine ---
        self.screen = None
        self.clock = pygame.time.Clock()
        self.draw_options = None
        self.reset()

    def reset(self, seed=None, options=None):
        """Resets the environment for a new training episode"""
        self.space = pymunk.Space()
        self.space.gravity = (0, 0)

        # Build Walls
        WALL_CATEGORY = 0b01
        rects = [
            [(50, 50), (WIDTH-50, 50)],
            [(WIDTH-50, 50), (WIDTH-50, HEIGHT-50)],
            [(WIDTH-50, HEIGHT-50), (50, HEIGHT-50)],
            [(50, HEIGHT-50), (50, 50)]
        ]
        for p1, p2 in rects:
            shape = pymunk.Segment(self.space.static_body, p1, p2, 5)
            shape.elasticity, shape.friction = 0.5, 0.5
            shape.filter = pymunk.ShapeFilter(categories=WALL_CATEGORY)
            self.space.add(shape)

        # Build Car
        self.car_mass = 1200
        self.car_width, self.car_height = 40, 20
        self.car_body = pymunk.Body(self.car_mass, pymunk.moment_for_box(self.car_mass, (self.car_width, self.car_height)))
        self.car_body.position = (WIDTH // 2, HEIGHT // 2)
        self.car_shape = pymunk.Poly.create_box(self.car_body, (self.car_width, self.car_height))
        self.car_shape.filter = pymunk.ShapeFilter(categories=0b10) # Ignore LiDAR
        self.space.add(self.car_body, self.car_shape)

        # Track AI progress
        self.steps = 0
        return self._get_obs(), {}

    def _get_obs(self):
        """Calculates LiDAR and speed for the AI to "see"."""
        readings = []
        start_x, start_y = self.car_body.position
        sensor_angles = [-math.pi/2, -math.pi/4, 0, math.pi/4, math.pi/2]
        sensor_length = 300

        # 1. LiDAR
        for angle in sensor_angles:
            ray_angle = self.car_body.angle + angle
            end_x = start_x + math.cos(ray_angle) * sensor_length
            end_y = start_y + math.sin(ray_angle) * sensor_length
            hit = self.space.segment_query_first((start_x, start_y), (end_x, end_y), 1, pymunk.ShapeFilter(mask=0b01))

            if hit: readings.append(hit.alpha * sensor_length)
            else: readings.append(sensor_length)

        # 2. Velocities
        forward_vec = pymunk.Vec2d(math.cos(self.car_body.angle), math.sin(self.car_body.angle))
        right_vec = pymunk.Vec2d(-math.sin(self.car_body.angle), math.cos(self.car_body.angle))

        self.f_vel = self.car_body.velocity.dot(forward_vec)
        self.l_vel = self.car_body.velocity.dot(right_vec)

        obs = np.array(readings + [self.f_vel, self.l_vel], dtype=np.float32)
        return obs

    def step(self, action):
        """Applies the AI's action and calculates the reward"""
        self.steps += 1
        steer, throttle, handbrake = action

        # Apply Throttle/Brake
        engine_power = 800000
        if throttle > 0: thrust = pymunk.Vec2d(engine_power * throttle, 0).rotated(self.car_body.angle)
        else: thrust = pymunk.Vec2d(engine_power * 0.5 * throttle, 0).rotated(self.car_body.angle)
        self.car_body.apply_force_at_local_point(thrust, (0, 0))

        # Apply Steering
        self.car_body.angle += float(steer * math.radians(4))
        self.car_body.angular_velocity *= 0.8

        # Apply Drift Physics
        grip = 0.98 if handbrake > 0 else 0.85
        forward_vec = pymunk.Vec2d(math.cos(self.car_body.angle), math.sin(self.car_body.angle))
        right_vec = pymunk.Vec2d(-math.sin(self.car_body.angle), math.cos(self.car_body.angle))

        f_vel_vec = forward_vec * self.car_body.velocity.dot(forward_vec)
        l_vel_vec = right_vec * self.car_body.velocity.dot(right_vec) * grip
        self.car_body.velocity = f_vel_vec + l_vel_vec

        # Step Physics
        self.space.step(1.0 / FPS)

        # ⚡ THE FIX: Force the environment to render itself natively every step
        self.render()

        # Calculate State & Reward
        obs = self._get_obs()
        min_lidar = min(obs[:5])

        reward = 0
        done = False

        # 1. Reward for moving forward fast
        reward += self.f_vel * 0.01

        # 2. Bonus Reward for drifting
        if abs(self.l_vel) > 50 and self.f_vel > 100:
            reward += abs(self.l_vel) * 0.05

        # 3. Massive penalty for crashing
        if min_lidar < 25:
            reward -= 500
            done = True

        # 4. Penalty for sitting still
        if self.steps > 100 and self.f_vel < 10:
            reward -= 10
            done = True

        return obs, reward, done, False, {}

    def render(self):
        """Draws the environment so we can watch the AI learn"""
        if self.screen is None:
            pygame.init()
            self.screen = pygame.display.set_mode((WIDTH, HEIGHT))
            self.draw_options = pymunk.pygame_util.DrawOptions(self.screen)

        pygame.event.get() # Keep Pygame responsive
        self.screen.fill((20, 20, 25))
        self.space.debug_draw(self.draw_options)
        pygame.display.flip()
        self.clock.tick(FPS)

# ==========================================
# 3. TRAINING LOOP
# ==========================================
if __name__ == "__main__":
    print("🚗 Initializing Autonomous Drift Agent...")
    env = DriftEnv()

    # Initialize the PPO Neural Network
    model = PPO("MlpPolicy", env, verbose=1)

    print("🧠 Training started. The AI is now in full control...")

    # ⚡ THE FIX: Hand complete control over to the SB3 backend for maximum speed
    model.learn(total_timesteps=100000)

    # Save the trained brain!
    model.save("drift_king_model")
    print("✅ Training complete and model saved.")


🚗 Initializing Autonomous Drift Agent...
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
🧠 Training started. The AI is now in full control...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 204      |
|    ep_rew_mean     | -213     |
| time/              |          |
|    fps             | 60       |
|    iterations      | 1        |
|    time_elapsed    | 34       |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 195          |
|    ep_rew_mean          | -205         |
| time/                   |              |
|    fps                  | 59           |
|    iterations           | 2            |
|    time_elapsed         | 69           |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0033922317 |

KeyboardInterrupt: 